# Tuning Optuna do TabM no Kaggle

Roda a busca de hiperparâmetros do TabM com Optuna nos 9 datasets do TabArena.

**Antes de rodar:** em *Session options* (painel direito) ative **Internet**.

**Como usar:**
1. Ajuste `N_TRIALS` na célula de configuração (padrão 50).
2. Clique em **Run All**.
3. Ao terminar, baixe `tuning_results_kaggle.csv` e `tuning_trials_kaggle.csv` da aba **Output**.
4. Copie esses arquivos para a pasta `results/` do repositório local.

> Os resultados são salvos **após cada dataset**, então uma interrupção não perde o progresso anterior.

In [ ]:
# ── 1. Setup ───────────────────────────────────────────────────────────────────
import os, sys, subprocess
from pathlib import Path

IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or Path('/kaggle/working').exists()

if IS_KAGGLE:
    subprocess.run(
        ['pip', 'install', '-q', 'pytabkit[models]', 'openml', 'optuna'],
        check=True
    )
    REPO_URL = 'https://github.com/aanasc4/projeto-am.git'
    REPO_DIR = Path('/kaggle/working/projeto-am')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
        print(f'Repositório clonado em {REPO_DIR}')
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
        print(f'Repositório atualizado em {REPO_DIR}')
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    RESULTS_DIR = Path('/kaggle/working/results')
    CACHE_DIR   = Path('/kaggle/working/cache/tabarena')
    os.environ['TABARENA_CACHE'] = str(CACHE_DIR)
else:
    sys.path.insert(0, str(Path('..').resolve()))
    RESULTS_DIR = Path('../results')
    CACHE_DIR   = Path('../cache/tabarena')

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Ambiente : {"Kaggle" if IS_KAGGLE else "Local"}')
print(f'Resultados: {RESULTS_DIR}')

In [ ]:
# ── 2. Configuração ────────────────────────────────────────────────────────────
N_TRIALS  = 50   # trials por dataset
CV_FOLDS  = 5
SEED      = 42

TASK_IDS_SMALL  = [10101, 37, 11]        # blood-transfusion, diabetes, balance-scale
TASK_IDS_MEDIUM = [31, 9952, 23]         # credit-g, phoneme, cmc
TASK_IDS_LARGE  = [7592, 14965, 146195]  # adult, bank-marketing, connect-4
TASK_IDS_ALL    = TASK_IDS_SMALL + TASK_IDS_MEDIUM + TASK_IDS_LARGE

BEST_PATH   = RESULTS_DIR / 'tuning_results_kaggle.csv'
TRIALS_PATH = RESULTS_DIR / 'tuning_trials_kaggle.csv'

print(f'N_TRIALS={N_TRIALS} | CV_FOLDS={CV_FOLDS} | SEED={SEED}')
print(f'{len(TASK_IDS_ALL)} datasets: {len(TASK_IDS_SMALL)} small + {len(TASK_IDS_MEDIUM)} medium + {len(TASK_IDS_LARGE)} large')

In [ ]:
# ── 3. Imports ─────────────────────────────────────────────────────────────────
import optuna
import pandas as pd

optuna.logging.set_verbosity(optuna.logging.WARNING)

from data.load_tabarena import load_task
from src.pipeline.split import stratified_split
from src.pipeline.tune import tabm_factory, tabm_search_space, tune

In [ ]:
# ── 4. Tuning com save incremental ─────────────────────────────────────────────
# Retoma de onde parou se os CSVs já existirem
if BEST_PATH.exists():
    done_ids = set(pd.read_csv(BEST_PATH)['task_id'].tolist())
    print(f'Retomando: {len(done_ids)} dataset(s) já prontos: {done_ids}')
else:
    done_ids = set()

for task_id in TASK_IDS_ALL:
    if task_id in done_ids:
        print(f'[task {task_id}] já processado, pulando.')
        continue

    ds = load_task(task_id)
    X_train, _, y_train, _ = stratified_split(ds.X, ds.y, seed=SEED)

    print(f'\n[{ds.name}]  n={ds.n_samples}, regime={ds.regime}, classes={ds.n_classes}')
    print(f'  Tuning: {N_TRIALS} trials × {CV_FOLDS}-fold CV ...')

    best_params, best_auc, study = tune(
        estimator_factory=tabm_factory,
        search_space=tabm_search_space,
        X=X_train, y=y_train,
        seed=SEED, n_trials=N_TRIALS, cv_folds=CV_FOLDS,
    )
    print(f'  Melhor AUC (CV)={best_auc:.4f} | params={best_params}')

    # Salva resultado do dataset imediatamente (append)
    best_row = pd.DataFrame([{
        'task_id': task_id, 'dataset': ds.name, 'regime': ds.regime,
        'best_auc_cv': best_auc, **best_params,
    }])
    best_row.to_csv(BEST_PATH, mode='a', header=not BEST_PATH.exists(), index=False)

    trial_rows = pd.DataFrame([{
        'task_id': task_id, 'dataset': ds.name,
        'trial': t.number, 'auc_cv': t.value, **t.params,
    } for t in study.trials])
    trial_rows.to_csv(TRIALS_PATH, mode='a', header=not TRIALS_PATH.exists(), index=False)

    print(f'  Salvo em {BEST_PATH}')

print('\nTuning concluído!')

In [ ]:
# ── 5. Resumo final ────────────────────────────────────────────────────────────
df = pd.read_csv(BEST_PATH)
print('Resultados por regime:')
print(df.groupby('regime')['best_auc_cv'].agg(['mean','min','max','count']).round(4).to_string())
print()
print(df[['dataset','regime','best_auc_cv','num_emb_type','tabm_k','n_epochs']].to_string(index=False))